# Group 8 Demo

**Project:** Code Defect Detection with Qwen, LoRA Fine-Tuning, and GraphCodeBERT  

**Demo introduction:**  
This project asks whether prompt-only large language models are enough for code defect detection, or whether task adaptation and code-specific pretraining are necessary.

## 1. Problem and Motivation

This section introduces the binary code defect detection task. Given a C/C++ function, the model predicts whether it is non-defective or defective. This matters because manual code review is expensive, defects can be subtle, and false negatives can leave vulnerable code undetected.

**Key point:** We are not only trying one model. We compare a ladder of methods from low-cost prompting to supervised code-specific modeling.

## 2. Research Question

This section states the main research question: for code defect detection, how far can prompt-only LLMs go, and when do fine-tuning or code-specific encoders become more effective?

The experiment design directly answers this question:

| Method | What it tests | Adaptation level |
|---|---|---|
| Qwen zero-shot | Raw prompt-only LLM reasoning | none |
| Qwen 4-shot | In-context examples without training | prompt examples |
| Qwen LoRA | Lightweight supervised LLM fine-tuning | LoRA adapter |
| Majority baseline | Sanity-check baseline | most frequent class |
| TF-IDF Linear SVM | Traditional lexical ML baseline | supervised CPU classifier |
| TF-IDF Logistic Regression | Traditional lexical ML baseline | supervised CPU classifier |
| GraphCodeBERT | Code-specific supervised encoder | full classifier head training |

## 3. Dataset and Evaluation

This section summarizes the dataset and metrics. We use the CodeXGLUE defect detection benchmark, where the label is binary: 0 for non-defective and 1 for defective. We report both validation and test results.

**Evaluation measures to mention:**

- Accuracy: overall correctness.
- Macro-F1: balanced score across both classes.
- Defective-F1 and defective recall: most important for whether the model actually catches defective code.
- Confusion matrix: shows false positives and false negatives directly.

Accuracy alone can be misleading here, because a model can look acceptable while missing the defective class. That is why the demo emphasizes Macro-F1 and Defective-F1.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
RESULTS = ROOT / "results"

metric_files = {
    ("Qwen zero-shot", "validation"): RESULTS / "zero_shot_validation_metrics.json",
    ("Qwen zero-shot", "test"): RESULTS / "zero_shot_test_metrics.json",
    ("Qwen 4-shot", "validation"): RESULTS / "four_shot_validation_metrics.json",
    ("Qwen 4-shot", "test"): RESULTS / "four_shot_test_metrics.json",
    ("Qwen LoRA fine-tuned", "validation"): RESULTS / "lora_validation_metrics.json",
    ("Qwen LoRA fine-tuned", "test"): RESULTS / "lora_test_metrics.json",
    ("Majority baseline", "test"): RESULTS / "majority_test_metrics.json",
    ("TF-IDF Linear SVM", "test"): RESULTS / "tfidf_linear_svm_test_metrics.json",
    ("TF-IDF Logistic Regression", "test"): RESULTS / "tfidf_logreg_test_metrics.json",
    ("GraphCodeBERT", "validation"): RESULTS / "graphcodebert_validation_metrics.json",
    ("GraphCodeBERT", "test"): RESULTS / "graphcodebert_test_metrics.json",
}

rows = []
for (method, split), path in metric_files.items():
    with path.open("r", encoding="utf-8") as f:
        m = json.load(f)
    rows.append({
        "method": method,
        "split": split,
        "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"],
        "defective_f1": m["defective_f1"],
        "defective_recall": m["defective_recall"],
        "tn": m["tn"],
        "fp": m["fp"],
        "fn": m["fn"],
        "tp": m["tp"],
    })

df = pd.DataFrame(rows)
print(df.sort_values(["split", "macro_f1"], ascending=[True, False]).to_string(index=False))

                    method      split  accuracy  macro_f1  defective_f1  defective_recall   tn  fp   fn  tp                                          file
             GraphCodeBERT       test    0.6589    0.6517        0.6017            0.5610 1096 381  551 704       results\graphcodebert_test_metrics.json
TF-IDF Logistic Regression       test    0.6223    0.6218        0.6088            0.6398  897 580  452 803        results\tfidf_logreg_test_metrics.json
         TF-IDF Linear SVM       test    0.6007    0.5994        0.5766            0.5920  898 579  512 743    results\tfidf_linear_svm_test_metrics.json
      Qwen LoRA fine-tuned       test    0.5523    0.5507        0.5236            0.5355  837 640  583 672                results\lora_test_metrics.json
            Qwen zero-shot       test    0.5264    0.5180        0.4545            0.4295  899 578  716 539           results\zero_shot_test_metrics.json
               Qwen 4-shot       test    0.5406    0.3509        0.0000     

## 4. Results Summary

This section summarizes all completed validation and test metrics. The table includes prompt-only Qwen, LoRA fine-tuning, traditional CPU baselines, and GraphCodeBERT.

| method | split | accuracy | macro_f1 | defective_f1 | defective_recall |
| --- | --- | --- | --- | --- | --- |
| GraphCodeBERT | test | 0.6589 | 0.6517 | 0.6017 | 0.5610 |
| TF-IDF Logistic Regression | test | 0.6223 | 0.6218 | 0.6088 | 0.6398 |
| TF-IDF Linear SVM | test | 0.6007 | 0.5994 | 0.5766 | 0.5920 |
| Qwen LoRA fine-tuned | test | 0.5523 | 0.5507 | 0.5236 | 0.5355 |
| Qwen zero-shot | test | 0.5264 | 0.5180 | 0.4545 | 0.4295 |
| Qwen 4-shot | test | 0.5406 | 0.3509 | 0.0000 | 0.0000 |
| Majority baseline | test | 0.5406 | 0.3509 | 0.0000 | 0.0000 |
| GraphCodeBERT | validation | 0.6618 | 0.6513 | 0.5908 | 0.5619 |
| Qwen LoRA fine-tuned | validation | 0.5556 | 0.5509 | 0.5045 | 0.5206 |
| Qwen zero-shot | validation | 0.5201 | 0.5086 | 0.4332 | 0.4221 |
| Qwen 4-shot | validation | 0.5655 | 0.3612 | 0.0000 | 0.0000 |

In [2]:
test_df = df[df["split"] == "test"].sort_values("macro_f1", ascending=False)
print(test_df[["method", "accuracy", "macro_f1", "defective_f1", "defective_recall", "tn", "fp", "fn", "tp"]].to_string(index=False))

                    method  accuracy  macro_f1  defective_f1  defective_recall   tn  fp   fn  tp
             GraphCodeBERT    0.6589    0.6517        0.6017            0.5610 1096 381  551 704
TF-IDF Logistic Regression    0.6223    0.6218        0.6088            0.6398  897 580  452 803
         TF-IDF Linear SVM    0.6007    0.5994        0.5766            0.5920  898 579  512 743
      Qwen LoRA fine-tuned    0.5523    0.5507        0.5236            0.5355  837 640  583 672
            Qwen zero-shot    0.5264    0.5180        0.4545            0.4295  899 578  716 539
               Qwen 4-shot    0.5406    0.3509        0.0000            0.0000 1477   0 1255   0
         Majority baseline    0.5406    0.3509        0.0000            0.0000 1477   0 1255   0


## 5. Final Test-Set Ranking

This section ranks models on the held-out test split. GraphCodeBERT has the strongest Macro-F1 at 0.6517. TF-IDF Logistic Regression is surprisingly competitive and has the best Defective-F1 at 0.6088. This means simple lexical baselines are strong, while the code-specific encoder is still the best balanced model overall.

| method | accuracy | macro_f1 | defective_f1 | defective_recall | tn | fp | fn | tp |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| GraphCodeBERT | 0.6589 | 0.6517 | 0.6017 | 0.5610 | 1096 | 381 | 551 | 704 |
| TF-IDF Logistic Regression | 0.6223 | 0.6218 | 0.6088 | 0.6398 | 897 | 580 | 452 | 803 |
| TF-IDF Linear SVM | 0.6007 | 0.5994 | 0.5766 | 0.5920 | 898 | 579 | 512 | 743 |
| Qwen LoRA fine-tuned | 0.5523 | 0.5507 | 0.5236 | 0.5355 | 837 | 640 | 583 | 672 |
| Qwen zero-shot | 0.5264 | 0.5180 | 0.4545 | 0.4295 | 899 | 578 | 716 | 539 |
| Qwen 4-shot | 0.5406 | 0.3509 | 0.0000 | 0.0000 | 1477 | 0 | 1255 | 0 |
| Majority baseline | 0.5406 | 0.3509 | 0.0000 | 0.0000 | 1477 | 0 | 1255 | 0 |

**Main takeaway sentence:**  
"For this task, supervised models work better than prompt-only LLMs, and the code-specific GraphCodeBERT model gives the best balanced performance."

## 6. Why 4-shot Collapsed

This section explains the 4-shot result. The result is not missing; it is a negative result. The 4-shot model predicted every test sample as non-defective, so its defective precision, recall, and F1 are all zero.

This shows that adding only four examples to the prompt was not enough for reliable code defect detection. The model learned a conservative output pattern instead of learning the defect signal. That supports the broader conclusion that prompt-only adaptation is weak for this task.

In [3]:
for _, row in test_df.iterrows():
    matrix = [[int(row["tn"]), int(row["fp"])], [int(row["fn"]), int(row["tp"])]]
    print(f"{row['method']} test confusion matrix [[TN, FP], [FN, TP]]:")
    print(matrix)
    print()

GraphCodeBERT test confusion matrix [[TN, FP], [FN, TP]]:
[[1096, 381], [551, 704]]

TF-IDF Logistic Regression test confusion matrix [[TN, FP], [FN, TP]]:
[[897, 580], [452, 803]]

TF-IDF Linear SVM test confusion matrix [[TN, FP], [FN, TP]]:
[[898, 579], [512, 743]]

Qwen LoRA fine-tuned test confusion matrix [[TN, FP], [FN, TP]]:
[[837, 640], [583, 672]]

Qwen zero-shot test confusion matrix [[TN, FP], [FN, TP]]:
[[899, 578], [716, 539]]

Qwen 4-shot test confusion matrix [[TN, FP], [FN, TP]]:
[[1477, 0], [1255, 0]]

Majority baseline test confusion matrix [[TN, FP], [FN, TP]]:
[[1477, 0], [1255, 0]]


## 7. Confusion Matrix Figure

This figure makes the model behavior visible. The 4-shot and majority baselines put everything into the non-defective column. TF-IDF Logistic Regression catches many defective examples, and GraphCodeBERT gives the best balanced Macro-F1.

![Confusion matrix overview](results/figures/confusion_matrices_overview.png)

## 8. Demo Walkthrough

This section identifies the main project artifacts. The code and outputs are organized in the GitHub repository: key scripts are in `scripts/`, configurations are in `configs/`, and final metrics and figures are in `results/`.

Demo artifact checklist:

1. `results/report_ready_metrics.md`: final metrics table.
2. `results/figures/confusion_matrices_overview.png`: visual comparison of model behavior.
3. `scripts/train_encoder_baseline.py` and `scripts/evaluate_encoder_baseline.py`: GraphCodeBERT training and evaluation.
4. `configs/encoder_graphcodebert.yaml`: GraphCodeBERT experiment configuration.
5. `demo.ipynb`: completed demo notebook that reads saved outputs.

In [4]:
artifact_paths = [
    RESULTS / "report_ready_metrics.md",
    RESULTS / "graphcodebert_test_metrics.json",
    RESULTS / "lora_test_metrics.json",
    RESULTS / "zero_shot_test_metrics.json",
    RESULTS / "four_shot_test_metrics.json",
    RESULTS / "figures" / "confusion_matrices_overview.png",
]

for path in artifact_paths:
    print(f"{path.relative_to(ROOT)}: {'OK' if path.exists() else 'MISSING'}")

results\report_ready_metrics.md: OK
results\graphcodebert_test_metrics.json: OK
results\lora_test_metrics.json: OK
results\zero_shot_test_metrics.json: OK
results\four_shot_test_metrics.json: OK
results\figures\confusion_matrices_overview.png: OK


## 9. Limitations and Future Work

This section frames the limitations. The project does not claim to solve defect detection completely. The best model is GraphCodeBERT, but its test Macro-F1 is about 0.65, so there is still room for improvement.

Future work:

- Better threshold tuning for defective recall.
- More code-specific baselines or larger encoder models.
- More careful prompt/example selection for few-shot prompting.
- Error analysis by defect type and code length.
- Try combining code-specific encoders with LLM explanations.

## 10. Closing Statement

The final conclusion is that prompt-only LLMs are not reliable enough for this code defect detection task. Traditional TF-IDF baselines are surprisingly strong, LoRA fine-tuning helps the LLM, and GraphCodeBERT gives the best balanced performance. This suggests that task supervision and code-aware modeling are both important for practical defect detection.

**Repo:** https://github.com/JiufuZh/526

## References

- CodeXGLUE benchmark: https://github.com/microsoft/CodeXGLUE
- GraphCodeBERT paper: https://arxiv.org/abs/2009.08366
- GraphCodeBERT model card: https://huggingface.co/microsoft/graphcodebert-base
- Qwen model family: https://huggingface.co/Qwen
- LoRA paper: https://arxiv.org/abs/2106.09685
- Project code: https://github.com/JiufuZh/526